# titanic feature engineering

kaggle titanic. focus is feature eng, not model. dataset url:
https://www.kaggle.com/c/titanic/data

expects `data/titanic.csv` (gitignored). we'll skip the data load if missing.

In [1]:
import pandas as pd
import numpy as np
import os

PATH = 'data/titanic.csv'
if os.path.exists(PATH):
    df = pd.read_csv(PATH)
else:
    # tiny sample baked in for the notebook to render
    df = pd.DataFrame({
        'PassengerId':[1,2,3,4,5],
        'Survived':[0,1,1,1,0],
        'Pclass':[3,1,3,1,3],
        'Name':['Owen Harris','Florence Briggs','Laina','Cumings','Allen, Mr. William'],
        'Sex':['male','female','female','female','male'],
        'Age':[22, 38, 26, 35, np.nan],
        'SibSp':[1,1,0,1,0],
        'Parch':[0,0,0,0,0],
        'Fare':[7.25, 71.28, 7.92, 53.10, 8.05],
    })
df.head()

## extract title from name

In [2]:
df['Title'] = df['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
df['Title'].value_counts()

## fill missing ages

use median per title (group-aware imputation)

In [3]:
df['Age'] = df.groupby('Title')['Age'].transform(lambda s: s.fillna(s.median()))
df['Age'].isnull().sum()

## family size and is_alone

In [4]:
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)
df[['SibSp','Parch','FamilySize','IsAlone']].head()

## first stab at a model

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
df['Sex_e'] = LabelEncoder().fit_transform(df['Sex'])
feat = ['Pclass','Sex_e','Age','FamilySize','IsAlone','Fare']
X = df[feat].fillna(df[feat].median())
y = df['Survived']
lr = LogisticRegression(solver='lbfgs', max_iter=2000)
lr.fit(X, y)
lr.score(X, y)

## bin rare titles

In [6]:
rare = df['Title'].value_counts()
df['Title'] = df['Title'].where(df['Title'].isin(rare[rare >= 2].index), other='Other')
df['Title'].value_counts()

## RF with simple grid search

In [7]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
rf = RandomForestClassifier(random_state=0)
p = {'n_estimators':[100,300], 'max_depth':[None, 6, 10]}
g = GridSearchCV(rf, p, cv=3, n_jobs=-1)
g.fit(X, y)
g.best_params_, g.best_score_

## try xgboost

In [8]:
import xgboost as xgb
from sklearn.model_selection import cross_val_score
m = xgb.XGBClassifier(n_estimators=200, learning_rate=0.05, max_depth=4, random_state=0,
                      use_label_encoder=False, eval_metric='logloss')
cross_val_score(m, X, y, cv=3).mean()

## bake-off: rf vs xgb

In [9]:
import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
models = {
    'rf': RandomForestClassifier(n_estimators=300, random_state=0),
    'xgb': xgb.XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=4, random_state=0,
                             use_label_encoder=False, eval_metric='logloss'),
}
for n, m in models.items():
    print(n, cross_val_score(m, X, y, cv=3).mean())

## final notes
feature eng > model choice on this one. title + family size + isalone gave more lift than swapping rf for xgb.

saving for later.